In [1]:
import jsonlines

In [2]:
import jsonlines

top_10_ptms = {
    'Phosphorylation': [],
    'Ubiquitination': [],
    'Acetylation': [],
    'N-linked Glycosylation': [],
    'Succinylation': []
}

for ptm in top_10_ptms.keys():
    top_10 = []                     # list of {id: score}
    top_count = 10
    id_to_idx = {}                  # maps ID -> index in top_10
    seen_site_logsum = set()        # set of (Site, LOGSUM) to drop exact duplicates
    counter = 0

    print(f"Checking for {ptm}")
    with jsonlines.open('pdb_ptm_scores.jsonl', mode='r') as reader:
        for line in reader:
            if counter % 10000 == 0 and counter != 0:
                print(f"\tDone with {counter} rows...")
            counter += 1

            for score in line[ptm]:
                curr_id = line['id']
                site = score['Site']
                logsum = float(score['LOGSUM'])

                # 1) Drop exact duplicates of (Site, LOGSUM)
                if (site, logsum) in seen_site_logsum:
                    continue

                # 2) If this ID is already in top_10, only keep the better LOGSUM
                if curr_id in id_to_idx:
                    idx = id_to_idx[curr_id]
                    existing_score = list(top_10[idx].values())[0]
                    existing_logsum = float(existing_score['LOGSUM'])

                    # If new score is better (less negative / higher), replace
                    if logsum > existing_logsum:
                        top_10[idx] = {curr_id: score}
                        # (We keep old (Site, LOGSUM) in seen_site_logsum so future exact duplicates are still dropped)
                    continue  # we've handled this ID; don't treat it as a "new" candidate

                # 3) ID is new and not a duplicate (Site, LOGSUM)
                if len(top_10) < top_count:
                    top_10.append({curr_id: score})
                    id_to_idx[curr_id] = len(top_10) - 1
                    seen_site_logsum.add((site, logsum))
                else:
                    # Find the current minimum (most negative) LOGSUM in top_10
                    min_idx = None
                    min_val = float('inf')
                    for i, item in enumerate(top_10):
                        # each item looks like {'some_id': {'Site': X, 'LOGSUM': Y}}
                        item_score = list(item.values())[0]
                        curr_val = float(item_score['LOGSUM'])
                        if curr_val < min_val:
                            min_val = curr_val
                            min_idx = i

                    # If new LOGSUM is greater (less negative) than the lowest in top_10, replace it
                    if logsum > min_val:
                        old_id = list(top_10[min_idx].keys())[0]
                        del id_to_idx[old_id]

                        top_10[min_idx] = {curr_id: score}
                        id_to_idx[curr_id] = min_idx
                        seen_site_logsum.add((site, logsum))

    # Sort the final top 10 by LOGSUM descending before storing
    top_10.sort(key=lambda x: float(list(x.values())[0]['LOGSUM']), reverse=True)
    top_10_ptms[ptm] = top_10


Checking for Phosphorylation
	Done with 10000 rows...
	Done with 20000 rows...
	Done with 30000 rows...
	Done with 40000 rows...
	Done with 50000 rows...
	Done with 60000 rows...
	Done with 70000 rows...
	Done with 80000 rows...
	Done with 90000 rows...
	Done with 100000 rows...
	Done with 110000 rows...
	Done with 120000 rows...
	Done with 130000 rows...
	Done with 140000 rows...
	Done with 150000 rows...
	Done with 160000 rows...
	Done with 170000 rows...
	Done with 180000 rows...
	Done with 190000 rows...
	Done with 200000 rows...
	Done with 210000 rows...
	Done with 220000 rows...
	Done with 230000 rows...
	Done with 240000 rows...
	Done with 250000 rows...
	Done with 260000 rows...
	Done with 270000 rows...
	Done with 280000 rows...
	Done with 290000 rows...
	Done with 300000 rows...
	Done with 310000 rows...
	Done with 320000 rows...
	Done with 330000 rows...
	Done with 340000 rows...
	Done with 350000 rows...
	Done with 360000 rows...
	Done with 370000 rows...
	Done with 380000 

In [3]:
# Optional: print summary
for ptm, values in top_10_ptms.items():
    print(f"\nTop 10 for {ptm}:")
    for v in values:
        print(v)


Top 10 for Phosphorylation:
{'5mqf_S': {'Site': 2542, 'LOGSUM': -40.80852243384422}}
{'5z57_U': {'Site': 2542, 'LOGSUM': -40.80852243384422}}
{'6hiv_BB': {'Site': 274, 'LOGSUM': -40.80852243384422}}
{'6ff4_S': {'Site': 2542, 'LOGSUM': -40.80852243384422}}
{'6hiv_BN': {'Site': 199, 'LOGSUM': -40.80852243384422}}
{'6ff7_S': {'Site': 2542, 'LOGSUM': -40.80852243384422}}
{'5xjc_U': {'Site': 2542, 'LOGSUM': -40.80852243384422}}
{'5yzg_U': {'Site': 2542, 'LOGSUM': -40.80852243384422}}
{'6hiv_BK': {'Site': 244, 'LOGSUM': -40.80852243384422}}
{'5z56_U': {'Site': 2542, 'LOGSUM': -40.80852243384422}}

Top 10 for Ubiquitination:
{'6dmp_A': {'Site': 60, 'LOGSUM': -48.55058421265694}}
{'6yvd_C': {'Site': 805, 'LOGSUM': -49.201267252625065}}
{'9d2f_K': {'Site': 137, 'LOGSUM': -49.25806748877136}}
{'3jbh_A': {'Site': 859, 'LOGSUM': -49.45228793981047}}
{'3jbh_B': {'Site': 859, 'LOGSUM': -49.45228793981047}}
{'8oyv_A': {'Site': 14, 'LOGSUM': -49.463244460274254}}
{'9ccf_A': {'Site': 111, 'LOGSUM': -4

In [ ]:
import json
for ptm, data in top_10_ptms.items():
    with open(f"result_dump_{ptm}.json", 'w') as f:
        json.dump(data, f, indent=2)